# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4 hoặc Kaggle T4)

Mã của em nằm trong `lab/` (notebook chỉ gọi các module này):

| File | Nội dung |
|---|---|
| `lab/common.py` | `PROMPT`, `yolo_to_box`, `objects_to_answer`, `compute_reward` (thuần Python) |
| `lab/vlm.py` | nạp Qwen3-VL-4B NF4, `encode_example` (mask loss), `collate`, `apply_setup`, `predict` |
| `lab/train.py` · `lab/infer.py` · `lab/grpo.py` | huấn luyện E1–E4 · suy luận · GRPO E5 |
| `lab/checks.py` · `lab/data_check.py` · `lab/viz.py` | kiểm tra bước 1–5 · vẽ box |
| `tools/kaggle_job.py` | chạy các lệnh GPU dưới dạng kernel Kaggle T4 x2 (2 làn song song, mỗi làn 1 GPU) |

Em chạy phần GPU trên **Kaggle (GPU T4 x2, 15 GB/GPU)** thay cho Colab; mỗi thí nghiệm dùng **một** T4 nên VRAM/thời gian tương đương Colab T4.

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Khoá Kaggle: Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY) hoặc biến môi trường KAGGLE_API_TOKEN từ .env (không commit).
# KHÔNG viết khoá vào notebook.
import os
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except ImportError:
    pass  # chạy local/Kaggle: đã có KAGGLE_API_TOKEN trong môi trường
!python -m pytest -q   # tín hiệu sẵn sàng: 68 passed, 1 skipped

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# Kiểm tra split theo phiên quay: số ảnh/box từng split, không phiên nào nằm ở hai split,
# và quy đổi YOLO -> JSON 0–1000 -> pixel khớp box gốc (mã trong lab/data_check.py).
!python -m lab.data_check
# Vẽ 6 ảnh train kèm box gốc (pothole đỏ, crack vàng, manhole xanh) -> report/figures/gt_samples_train.jpg
!python -m lab.viz gt --split train --n 6 --seed 10 --out report/figures/gt_samples_train.jpg
from IPython.display import Image as _Img
_Img("report/figures/gt_samples_train.jpg", width=900)

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# yolo_to_box(), objects_to_answer() và PROMPT nằm trong lab/common.py (dùng chung cho train/infer/GRPO).
import json
from lab.common import PROMPT, objects_to_answer, yolo_to_box, box_to_pixel, read_jsonl

ex = read_jsonl("data/splits/train.jsonl")[0]
ans = objects_to_answer(ex["objects"])        # JSON gọn, box sắp theo (y1, x1)
print(PROMPT)
print(ans)
# quy ngược về pixel để so với box YOLO gốc
for b in json.loads(ans)[:3]:
    print(b["label"], b["bbox_2d"], "->", [round(v, 1) for v in box_to_pixel(b["bbox_2d"], 640, 360)])

## 3. E0: zero-shot

In [ ]:
# Nạp Qwen3-VL-4B NF4 (llm_int8_skip_modules=["visual","lm_head"]) và suy luận zero-shot 311 ảnh test, batch 8,
# greedy, max_new_tokens=256 (configs/e0.yaml). Mã: lab/vlm.py (load_model, predict) + lab/infer.py.
!python -m lab.infer --out work/e0.jsonl
# Trên Kaggle (em dùng):  python tools/kaggle_job.py push --slug lab10-main --lane "python -m lab.infer --out work/e0.jsonl ;; ..."

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# encode_example(): áp chat template 2 lần (chỉ user + add_generation_prompt / user + assistant),
# kiểm tra prompt là tiền tố, labels[:n_prompt] = -100 (token ảnh + prompt), bỏ "\n" sau <|im_end|>.
# collate(): pad input_ids/attention_mask/mm_token_type_ids/labels(-100); pixel_values, image_grid_thw nối theo chiều 0.
# Kiểm tra: 640x360 -> image_grid_thw [1,22,40] (patch 16, merge 2) -> 220 token ảnh; giải mã labels != -100 = JSON + <|im_end|>.
!python -m lab.checks
import inspect
from lab import vlm
print(inspect.getsource(vlm.encode_example))
print(inspect.getsource(vlm.collate))

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# apply_setup(): LoRA theo regex trên tên module thật của Qwen3-VL-4B
#   LLM : model.language_model.layers.N.self_attn.{q,k,v,o}_proj + mlp.{gate,up,down}_proj  -> 36 x 7 = 252 module
#   ViT : model.visual.blocks.N.attn.{qkv,proj} + mlp.{linear_fc1,linear_fc2}                -> 24 x 4 = 96 module (chỉ E3)
#   E2/E3: mở full visual.merger + visual.deepstack_merger_list (109.1M tham số) và đưa về float32
print(inspect.getsource(vlm.apply_setup))
import json
print(json.dumps(json.load(open("report/checks.json"))["trainable"], indent=1))

## 6. Vòng huấn luyện

In [ ]:
# Vòng huấn luyện (lab/train.py): AdamW lr 2e-4 wd 0, warmup 5% + cosine, batch 2 x tích luỹ 4, 300 bước,
# autocast fp16 + GradScaler, clip grad 1.0, gradient checkpointing (use_reentrant=False), use_cache=False.
# Mỗi 100 bước lưu adapter VÀ projector.pt (E2/E3); cuối cùng ghi run.json (trainable_params, peak_vram_gb, train_minutes).
!python -m lab.train --setup E1 --out work/E1
!python -m lab.train --setup E2 --out work/E2
!python -m lab.train --setup E3 --out work/E3

## 7. Suy luận và chấm E1–E3

In [ ]:
# Nạp model gốc 4-bit -> adapter LoRA -> nạp lại projector.pt (E2/E3) -> suy luận test -> chấm kèm meta từ run.json
import json, subprocess
for n, setup in ((1, "E1"), (2, "E2"), (3, "E3")):
    proj = f"--projector work/{setup}/projector.pt" if setup != "E1" else ""
    !python -m lab.infer --adapter work/{setup}/adapter {proj} --out work/e{n}.jsonl
    run = json.load(open(f"work/{setup}/run.json"))
    !python scripts/score.py --pred work/e{n}.jsonl --name {setup} --out report/evidence/e{n}.json \
        --meta trainable_params={run['trainable_params']} --meta peak_vram_gb={run['peak_vram_gb']} \
        --meta train_minutes={run['train_minutes']}

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4 — ablation MỘT biến: data_size (train_fraction 0.1 = 142 ảnh vs 1.0 = 1.422 ảnh),
# giữ nguyên mọi thứ khác của setup tốt nhất (cùng r, lr, batch, 300 bước, seed).
# Giá trị 1.0 chính là run gốc của setup đó -> chấm lại cùng file dự đoán dưới tên e4_data_1.0.
!python -m lab.train --setup E1 --out work/E4_data_0.1 --train-fraction 0.1
!python -m lab.infer --adapter work/E4_data_0.1/adapter --out work/e4_data_0.1.jsonl

# E5 (tuỳ chọn) — GRPO với reward IoU (lab/common.py: compute_reward; test: lab/tests/test_reward.py)
!python -m pytest lab/tests -q
!python -m lab.grpo --init work/E1 --out work/E5          # 60 bước, 4 lần sinh/ảnh, lr 5e-6, beta 0
!python -m lab.infer --adapter work/E5/adapter --out work/e5.jsonl

# Bảng tổng hợp lấy thẳng từ report/evidence/*.json và phân tích lỗi
!python -m lab.summarize
!python -m lab.analyze --pred work/e1.jsonl --name E1 --figs